In [2]:
import json

def merge_content_back(original_jsonl: str, llm_output_jsonl: str, final_workbook_jsonl: str):
    # 1. Load the original content into a dictionary mapped by article_id
    content_map = {}
    with open(original_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                content_map[data["article_id"]] = data["content"]

    # 2. Read LLM results and inject the content downstream
    matched_count = 0
    with open(llm_output_jsonl, 'r', encoding='utf-8') as infile, \
         open(final_workbook_jsonl, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            if line.strip():
                result = json.loads(line)
                art_id = result.get("article_id")
                
                # Pull the content from memory instead of the LLM payload
                if art_id in content_map:
                    result["content"] = content_map[art_id]
                    matched_count += 1
                
                outfile.write(json.dumps(result, ensure_ascii=False) + '\n')

    print(f"Successfully stitched content into {matched_count} records for human annotation.")

merge_content_back(
        original_jsonl='llm_rab.jsonl', 
        llm_output_jsonl='gemini3.1_pro_ext_rab.jsonl', 
        final_workbook_jsonl='gold_standard_rab.jsonl'
    )

Successfully stitched content into 8 records for human annotation.
